----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Discovery layer - Convert Symbols
This notebook covers the public APIs `convert_symbols`, `SymbolTypes`, `AssetClass`, `AssetState` and `CountryCode`.
`convert_symbols` maps identifiers and returns a pandas DataFrame. `SymbolTypes` describes source and target identifiers, `AssetClass` and `AssetState` constrain matches, and `CountryCode` supplies a country preference. Mappings are not necessarily one-to-one; review the returned instruments and identifiers rather than assuming a unique match.

**Prerequisites:** a configured session entitled to symbol conversion. Keep credentials in the library configuration. Identifier availability and mapping results can change; errors are not suppressed.

#### Learn more
Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials.

#### Getting Help and Support
For API questions, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory.

In [ ]:
import os

os.environ["LD_LIB_CONFIG_PATH"] = os.path.join("..", "..", "Configuration")

## Some Imports to start with

In [ ]:
import lseg.data as ld
from lseg.data.discovery import convert_symbols, SymbolTypes, AssetClass, AssetState, CountryCode

## Open the data session
Use the default session in `lseg-data.config.json`.

In [ ]:
ld.open_session()

## Convert one RIC to multiple identifier types
The test converts `IBM.N` to an organization PermID and ticker, preferring the USA. The result includes `DocumentTitle`, `TickerSymbol` and `IssuerOAPermID`.

The test passes `preferred_country_code="USA"`; the equivalent public enum member is `CountryCode.USA`. Prefer the member over manually copying its internal service value.

In [ ]:
ibm = convert_symbols(
    symbols="IBM.N",
    from_symbol_type=SymbolTypes.RIC,
    to_symbol_types=[SymbolTypes.OA_PERM_ID, SymbolTypes.TICKER_SYMBOL],
    preferred_country_code=CountryCode.USA,
)
ibm

## Convert several RICs to the default target types
Omitting `to_symbol_types` requests the default set of target identifiers. The test uses the asset classes below and `AssetState.ACTIVE`. Typical returned columns include `RIC`, `DocumentTitle`, `SEDOL`, `TickerSymbol`, `IssueISIN`, `IssuerOAPermID` and `CUSIP`. Some mappings may be unavailable.

In [ ]:
converted = convert_symbols(
    symbols=["AAPL.O", "GOOG.O", "IBM.N"],
    from_symbol_type=SymbolTypes.RIC,
    asset_class=[
        AssetClass.COMMODITIES, AssetClass.EQUITIES, AssetClass.BONDS,
        AssetClass.FUNDS, AssetClass.WARRANTS, AssetClass.FX_AND_MONEY,
    ],
    asset_state=AssetState.ACTIVE,
)
converted

## Inspect the public enumeration choices
These members are available locally without a service request. Use `from_symbol_type` when the source identifier type is known; when omitted, the library uses its default unique-identifier scope.

In [ ]:
print("Symbol types:", [member.name for member in SymbolTypes])
print("Asset classes:", [member.name for member in AssetClass])
print("Asset states:", [member.name for member in AssetState])

## Close the session
The test also verifies that conversion with a closed session raises `ValueError`. Close only this notebook's session after completing the examples; do not continue making data requests without reopening it. No server-side resources or local files were created.

In [ ]:
ld.close_session()